# Checkpoint B4: Self-Check — Tự kiểm tra kết quả trích xuất

Checkpoint này triển khai hàm `run_self_check()` để tự động kiểm tra chất lượng JSON output trước khi chấp nhận kết quả.

**Điều kiện đạt:**
- [x] Schema check: đủ trường bắt buộc, đúng kiểu dữ liệu
- [x] Source evidence check: mỗi trường có trích dẫn, trích dẫn tồn tại trong gốc
- [x] Confidence calibration: điều chỉnh confidence dựa trên số evidence
- [x] Trả về: passed, errors[], warnings[], adjusted_confidence

**Tiếp theo:** Sau khi pass, chuyển sang Checkpoint C3 (red-flag checking).

In [4]:
import json
import re
from typing import Any

# ============================================================
# Hàm self-check: Kiểm tra chất lượng JSON output
# ============================================================

# Các trường bắt buộc theo schema và kiểu dữ liệu kỳ vọng
REQUIRED_FIELDS = {
    "contract_id": str,
    "effective_date": (str, type(None)),    # string hoặc null
    "expiry_date": (str, type(None)),
    "penalty_clause": str,
    "source_evidence": list,
    "confidence": (int, float),
    "needs_human_review": bool,
    "red_flags": list,
    "missing_fields": list,
    "extraction_notes": str,
}


def run_self_check(extracted_json: dict, contract_text: str) -> dict:
    """Tự kiểm tra kết quả trích xuất JSON dựa trên 3 lớp validation.

    Lớp 1 — Schema check:
    - Kiểm tra đủ trường bắt buộc
    - Kiểm tra đúng kiểu dữ liệu

    Lớp 2 — Source evidence check:
    - Mỗi source_evidence phải có đủ field, quote, section
    - Trích dẫn (quote) phải tồn tại trong văn bản gốc ( fuzzy match )

    Lớp 3 — Confidence calibration:
    - So sánh reported confidence với số lượng evidence items
    - Điều chỉnh nếu confidence quá cao mà ít evidence

    Args:
        extracted_json: Kết quả trích xuất từ LLM (dict)
        contract_text: Văn bản gốc hợp đồng (để kiểm tra trích dẫn)

    Returns:
        Dict chứa:
          - passed (bool): True nếu không có errors
          - errors (list[str]): Các lỗi nghiêm trọng
          - warnings (list[str]): Các cảnh báo
          - adjusted_confidence (float): Confidence sau khi điều chỉnh
    """
    errors = []
    warnings = []

    # ──────────────────────────────────────
    # LỚP 1: Schema check — đủ trường + đúng kiểu
    # ──────────────────────────────────────
    for field, expected_type in REQUIRED_FIELDS.items():
        if field not in extracted_json:
            errors.append(f"Schema: thiếu trường bắt buộc '{field}'")
        else:
            value = extracted_json[field]
            if not isinstance(value, expected_type):
                actual = type(value).__name__
                expected = expected_type.__name__ if isinstance(expected_type, type) else str(expected_type)
                errors.append(f"Schema: trường '{field}' sai kiểu — kỳ vọng {expected}, thực tế {actual}")

    # Nếu schema lỗi nặng thì dừng sớm
    if errors:
        return {
            "passed": False,
            "errors": errors,
            "warnings": warnings,
            "adjusted_confidence": 0.0,
        }

    # ──────────────────────────────────────
    # LỚP 2: Source evidence check
    # ──────────────────────────────────────
    evidence_list = extracted_json.get("source_evidence", [])
    text_lower = contract_text.lower()

    if len(evidence_list) == 0:
        warnings.append("Evidence: không có source_evidence nào — không thể xác minh trích xuất")
    else:
        for i, ev in enumerate(evidence_list):
            # Kiểm tra cấu trúc mỗi evidence item
            if not all(k in ev for k in ("field", "quote", "section")):
                errors.append(f"Evidence[{i}]: thiếu field/quote/section")
                continue

            # Kiểm tra quote có tồn tại trong văn bản gốc (fuzzy: lowercase, bỏ dấu)
            quote = ev["quote"]
            # Tìm kiếm fuzzy: dùng 10 ký tự đầu tiên của quote
            snippet = quote[:30].lower().strip()
            if snippet and snippet not in text_lower:
                warnings.append(f"Evidence[{i}] ({ev['field']}): trích dẫn không khớp chính xác trong văn bản gốc")
                # Không mark lỗi — có thể do OCR khác biệt

    # ──────────────────────────────────────
    # LỚP 3: Confidence calibration
    # ──────────────────────────────────────
    reported_confidence = extracted_json.get("confidence", 0.0)
    num_evidence = len(evidence_list)
    num_missing = len(extracted_json.get("missing_fields", []))

    # Công thức điều chỉnh:
    # Mỗi evidence item đóng góp ~0.1 confidence (tối đa 1.0)
    # Mỗi missing field trừ 0.15
    evidence_score = min(num_evidence * 0.1, 1.0)
    missing_penalty = num_missing * 0.15
    adjusted_confidence = round(max(0.0, min(1.0, evidence_score - missing_penalty)), 2)

    # So sánh reported vs adjusted
    diff = abs(reported_confidence - adjusted_confidence)
    if diff > 0.2:
        warnings.append(
            f"Confidence: reported={reported_confidence}, adjusted={adjusted_confidence} "
            f"(chênh {diff:.2f} — có thể không phản ánh đúng căn cứ)"
        )

    # Nếu confidence adjusted thấp mà không bật HITL → cảnh báo
    if adjusted_confidence < 0.7 and not extracted_json.get("needs_human_review", False):
        warnings.append("Confidence adjusted < 0.7 nhưng needs_human_review=False — nên bật HITL")

    # ──────────────────────────────────────
    # KẾT QUẢ
    # ──────────────────────────────────────
    passed = len(errors) == 0

    return {
        "passed": passed,
        "errors": errors,
        "warnings": warnings,
        "adjusted_confidence": adjusted_confidence,
    }


print("run_self_check() đã định nghĩa sẵn")
print("Gọi: result = run_self_check(extracted_json, contract_text)")

run_self_check() đã định nghĩa sẵn
Gọi: result = run_self_check(extracted_json, contract_text)


In [5]:
# ============================================================
# Test self-check với sample extracted JSON
# ============================================================

# --- Dữ liệu mẫu: JSON trích xuất từ contract-001 (đầy đủ, hợp lệ) ---
sample_extracted = {
    "contract_id": "contract-001",
    "effective_date": "2026-01-01",
    "expiry_date": "2026-12-31",
    "penalty_clause": "Phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất",
    "source_evidence": [
        {"field": "effective_date", "quote": "Hợp đồng có hiệu lực kể từ ngày 01 tháng 01 năm 2026", "section": "Điều 2"},
        {"field": "expiry_date", "quote": "Hợp đồng có thời hạn 12 tháng, hết hạn vào ngày 31 tháng 12 năm 2026", "section": "Điều 2"},
        {"field": "penalty_clause", "quote": "Phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất", "section": "Điều 4"},
    ],
    "confidence": 0.92,
    "needs_human_review": False,
    "red_flags": [],
    "missing_fields": [],
    "extraction_notes": "Hợp đồng đầy đủ, rõ ràng.",
}

# Văn bản gốc contract-001 (rút gọn để test)
sample_contract_text = """
HỢP ĐỒNG CUNG CẤP DỊCH VỤ VIỄN THÔNG
ĐIỀU 2: Thời hạn hợp đồng
Hợp đồng có hiệu lực kể từ ngày 01 tháng 01 năm 2026 đến ngày 31 tháng 12 năm 2026.
Hợp đồng có thời hạn 12 tháng, hết hạn vào ngày 31 tháng 12 năm 2026.
ĐIỀU 4: Phạt vi phạm SLA
Phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất. Tối đa phạt không quá 10%.
"""

# --- Chạy self-check ---
print("=" * 60)
print("SELF-CHECK: contract-001 (đầy đủ, hợp lệ)")
print("=" * 60)
check_result = run_self_check(sample_extracted, sample_contract_text)
print(f"Passed: {check_result['passed']}")
print(f"Adjusted confidence: {check_result['adjusted_confidence']}")
if check_result['errors']:
    print("Errors:")
    for e in check_result['errors']:
        print(f"  - {e}")
if check_result['warnings']:
    print("Warnings:")
    for w in check_result['warnings']:
        print(f"  - {w}")
else:
    print("No errors or warnings.")

# --- Test case 2: JSON thiếu trường (contract-002) ---
print("\n" + "=" * 60)
print("SELF-CHECK: contract-002 (thiếu trường)")
print("=" * 60)

sample_extracted_bad = {
    "contract_id": "contract-002",
    "effective_date": None,
    "expiry_date": None,
    "penalty_clause": "Phạt 0.5% cho mỗi tuần",
    "source_evidence": [],
    "confidence": 0.92,  # Confidence cao nhưng không có evidence!
    "needs_human_review": False,  # Sai — nên là True
    "red_flags": [],
    "missing_fields": ["expiry_date", "confidentiality"],
    "extraction_notes": "Điều 2 bị lỗi OCR.",
}

check_bad = run_self_check(sample_extracted_bad, "[Văn bản mẫu]")
print(f"Passed: {check_bad['passed']}")
print(f"Adjusted confidence: {check_bad['adjusted_confidence']}")
if check_bad['warnings']:
    print("Warnings:")
    for w in check_bad['warnings']:
        print(f"  - {w}")

# --- Test case 3: JSON thiếu trường bắt buộc ---
print("\n" + "=" * 60)
print("SELF-CHECK: JSON thiếu trường bắt buộc")
print("=" * 60)

sample_missing_field = {
    "contract_id": "contract-003",
    # Thiếu effective_date, expiry_date, source_evidence, v.v.
    "confidence": 0.5,
}

check_missing = run_self_check(sample_missing_field, "[Văn bản mẫu]")
print(f"Passed: {check_missing['passed']}")
if check_missing['errors']:
    print("Errors:")
    for e in check_missing['errors']:
        print(f"  - {e}")

SELF-CHECK: contract-001 (đầy đủ, hợp lệ)
Passed: True
Adjusted confidence: 0.3
Warnings:
  - Confidence: reported=0.92, adjusted=0.3 (chênh 0.62 — có thể không phản ánh đúng căn cứ)
  - Confidence adjusted < 0.7 nhưng needs_human_review=False — nên bật HITL

SELF-CHECK: contract-002 (thiếu trường)
Passed: True
Adjusted confidence: 0.0
Warnings:
  - Evidence: không có source_evidence nào — không thể xác minh trích xuất
  - Confidence: reported=0.92, adjusted=0.0 (chênh 0.92 — có thể không phản ánh đúng căn cứ)
  - Confidence adjusted < 0.7 nhưng needs_human_review=False — nên bật HITL

SELF-CHECK: JSON thiếu trường bắt buộc
Passed: False
Errors:
  - Schema: thiếu trường bắt buộc 'effective_date'
  - Schema: thiếu trường bắt buộc 'expiry_date'
  - Schema: thiếu trường bắt buộc 'penalty_clause'
  - Schema: thiếu trường bắt buộc 'source_evidence'
  - Schema: thiếu trường bắt buộc 'needs_human_review'
  - Schema: thiếu trường bắt buộc 'red_flags'
  - Schema: thiếu trường bắt buộc 'missing_

In [6]:
# ============================================================
# Ghi file script python validator.py chuẩn vào thư mục scripts/
# ============================================================
import shutil
import os

PROJECT_ROOT = "../../outputs/contract-term-extractor"
src_script = "../../templates/skills/contract-term-extractor/scripts/validator.py"
dst_script = f"{PROJECT_ROOT}/scripts/validator.py"

os.makedirs(os.path.dirname(dst_script), exist_ok=True)
shutil.copy(src_script, dst_script)
print(f"Đã copy validator.py thành công vào: {dst_script}")

Đã copy validator.py thành công vào: ../../outputs/contract-term-extractor/scripts/validator.py
